# Erosita Re-import

- Author: Melissa D.
- Last Run: Feb 9, 2026
- https://github.com/astronomy-commons/data.lsdb.io/issues/171

Fetch the raw data again:

```
$ wget https://erosita.mpe.mpg.de/dr1/AllSkySurveyData_dr1/Catalogues_dr1/MerloniA_DR1/eRASS1_Main.v1.2.fits.tar.gz
$ tar -xvzf eRASS1_Main.v1.2.fits.tar.gz
```

In [1]:
import hats
import numpy as np
from dask.distributed import Client
from hats_import import CollectionArguments, pipeline_with_client, pipeline, VerificationArguments, ImportArguments
from pathlib import Path
from hats_import.catalog.file_readers import CsvReader
from astropy.io import ascii
import lsdb
import pandas as pd

hats.__version__

'0.8.1'

In [2]:
single_file = Path("/epyc/data3/hats/raw/erass/eRASS1_Main.v1.2.fits")

In [3]:
args = (
    CollectionArguments(
        completion_email_address="delucchi@andrew.cmu.edu",
        output_artifact_name="erass_dr1",
        output_path="/epyc/data3/hats/catalogs/v06",
        progress_bar=True,
        simple_progress_bar=True,
    )
    .catalog(
        output_artifact_name="erass_dr1",
        input_file_list=[single_file],
        file_reader="fits",
        ra_column='RA',
        dec_column='DEC',
        expected_total_rows=930_203,
        pixel_threshold=1_000_000,
        highest_healpix_order=8,
        skymap_alt_orders=[2, 4, 6],
        row_group_kwargs={"num_rows": 50_000},
    )
    .add_margin(margin_threshold=5.0, is_default=True)
)

In [4]:
with Client(
    local_directory="/epyc/data3/hats/tmp/",
    n_workers=2,
    threads_per_worker=1,
) as client:
    pipeline_with_client(args, client)

Margin: Finishing :   0%|          | 0/4 [00:00<?, ?it/s]/astro/users/mmd11/.conda/envs/lsdbv08/lib/python3.12/site-packages/hats/catalog/partition_info.py:113: UserWarning: Computing partitions from catalog parquet files. This may be slow.
  warnings.warn("Computing partitions from catalog parquet files. This may be slow.")
Collection: Finishing : 100%|██████████| 2/2 [00:00<00:00, 254.68it/s]


In [5]:
args = VerificationArguments(
    input_catalog_path="/epyc/data3/hats/catalogs/v06/erass_dr1",
    output_path="./verification/erass_dr1",
)
pipeline(args)

Loading dataset and schema.

Starting: Test hats.io.validation.is_valid_collection.
Validating collection at path /epyc/data3/hats/catalogs/v06/erass_dr1 ... 
Validating catalog at path /epyc/data3/hats/catalogs/v06/erass_dr1/erass_dr1 ... 
Found 10 partitions.
Approximate coverage is 77.08 % of the sky.
Validating catalog at path /epyc/data3/hats/catalogs/v06/erass_dr1/erass_dr1_5arcs ... 
Found 10 partitions.
Approximate coverage is 77.08 % of the sky.
Result: PASSED

Starting: Test that files in _metadata match the data files on disk.
Result: PASSED

Starting: Test that number of rows are equal.
	file footers vs catalog properties
	file footers vs _metadata
Result: PASSED

Starting: Test that schemas are equal, excluding metadata.
	_common_metadata vs truth
	_metadata vs truth
	file footers vs truth
Result: PASSED

Verifier results written to verification/erass_dr1/verifier_results.csv
Elapsed time (seconds): 0.09
